Módulo desenvolvido para processar e extrair séries horárias de temperatura
(T2 – temperatura a 2 metros) a partir de arquivos mensais do modelo WRF
(Weather Research and Forecasting), disponibilizados em formato NetCDF (.nc).
-----------------------------------------------------------------------------

O objetivo principal deste módulo é gerar, para cada município brasileiro,
uma série temporal horária contendo a temperatura média dos pixels do WRF
que cruzam os limites municipais, permitindo sua integração com modelos e
inventários atmosféricos que dependem de temperatura local — como emissões
evaporativas de combustíveis.

Autor: Marcos Perrude  
Data: 09 de outubro de 2025

In [1]:
import geopandas as gpd
import pandas as pd
import os
import xarray as xr
from tqdm import tqdm   # barra de progresso
import glob
from functions.processarMunicipio import processar_municipio
from functions.lerTemperaturaWRF import ler_temperatura_wrf
from functions.netcdf4_conversions_v2 import brain_to_latlng

from multiprocessing import Pool
from functools import partial

In [2]:
############### VARIAVEIS
# Caminho geral
tablePath = os.path.dirname(os.path.dirname(os.getcwd()))

# Caminho das pastas
dataPath = tablePath + '/inputs'
outPath = tablePath + "/outputs"
outPathInter = tablePath + "/outputs_intermediarios"

# SHP municípios IBGE
shp_mun =  dataPath + '/BR_Municipios_2022/BR_Municipios_2022.shp'

# Arquivos METCRO2D
metcroPath = '/home/artaxo/marcos/saldiva_mnt/mcip/BR_12km'

# COLOCAR OS ANOS QUE VÃO SER ANALISADOS
anos = [2023]
meses = [1,2,3,4,5,6,7,8,9,10,11,12]

# NÚEMRO DE NÚCLEOS
n_processadores = 30

# Pasta raiz onde estao os dados mensais do WRF
#path_temp = dataPath + "/WRF/temp_month/"
# Lista dos arquivos
#arquivos_temp = sorted(glob.glob(os.path.join(dataPath + "/WRF/temp_month/", "*_T2.nc")))

In [15]:
################### CARREGAMENTO

# SHP municipios IBGE
shp_mun = gpd.read_file(shp_mun).to_crs("EPSG:4326")
shp_mun['CD_MUN'] = shp_mun['CD_MUN'].astype(int)

In [4]:
for mes in meses:
    arquivos = sorted([os.path.join(metcroPath, f)for f in os.listdir(os.path.join(metcroPath))
                               if f.startswith('METCRO2D_BR_12km_')and f'{2023}-{mes:02d}' in f])

In [5]:
arquivos

['/home/artaxo/marcos/saldiva_mnt/mcip/BR_12km/METCRO2D_BR_12km_2023-12-01.nc',
 '/home/artaxo/marcos/saldiva_mnt/mcip/BR_12km/METCRO2D_BR_12km_2023-12-02.nc',
 '/home/artaxo/marcos/saldiva_mnt/mcip/BR_12km/METCRO2D_BR_12km_2023-12-03.nc',
 '/home/artaxo/marcos/saldiva_mnt/mcip/BR_12km/METCRO2D_BR_12km_2023-12-04.nc',
 '/home/artaxo/marcos/saldiva_mnt/mcip/BR_12km/METCRO2D_BR_12km_2023-12-05.nc',
 '/home/artaxo/marcos/saldiva_mnt/mcip/BR_12km/METCRO2D_BR_12km_2023-12-06.nc',
 '/home/artaxo/marcos/saldiva_mnt/mcip/BR_12km/METCRO2D_BR_12km_2023-12-07.nc',
 '/home/artaxo/marcos/saldiva_mnt/mcip/BR_12km/METCRO2D_BR_12km_2023-12-08.nc',
 '/home/artaxo/marcos/saldiva_mnt/mcip/BR_12km/METCRO2D_BR_12km_2023-12-09.nc',
 '/home/artaxo/marcos/saldiva_mnt/mcip/BR_12km/METCRO2D_BR_12km_2023-12-10.nc',
 '/home/artaxo/marcos/saldiva_mnt/mcip/BR_12km/METCRO2D_BR_12km_2023-12-11.nc',
 '/home/artaxo/marcos/saldiva_mnt/mcip/BR_12km/METCRO2D_BR_12km_2023-12-12.nc',
 '/home/artaxo/marcos/saldiva_mnt/mcip/B

In [5]:

# ano = 2023
# mes = 1
# arquivos = sorted([os.path.join(metcroPath, f)for f in os.listdir(os.path.join(metcroPath))
#                            if f.startswith('METCRO2D_BR_12km_')and f'{ano}-{mes:02d}' in f])
# T2_xr = ler_temperatura_wrf(arquivos,shp_mun)

In [7]:
# municipios = [mun for _, mun in shp_mun.iterrows()]
# municipios[0]

In [ ]:
############# PROCESSAMENTO

# COLOCAR OS ANOS ANALISADOS
for ano in anos:
    for mes in meses:
        print(f"\n>>> Lendo arquivo: {mes}-{ano}")
        
        # Camiho para a pasta do WRF do ano e mes analisado
        # metcroPath = f'/mnt/storage_local_data/GAr_BR/BR_12km/WRF/{ano}/{ano}_{mes:02d}'
        
        # Listagem dos arquivos diários para apenas o mes analisado
        #arquivos = sorted([os.path.join(metcroPath, f) for f in os.listdir(metcroPath) if f.startswith('wrfout_d02') and f'{ano}-{mes:02d}' in f])

        arquivos = sorted([os.path.join(metcroPath, f)for f in os.listdir(os.path.join(metcroPath))
                           if f.startswith('METCRO2D_BR_12km_')and f'{ano}-{mes:02d}' in f])
        # Extraindo a temepratura média horário do WRF
        T2_xr = ler_temperatura_wrf(arquivos,shp_mun)
    
        # Define o início e fim do mês automaticamente
        inicio_mes = f"{ano}-{mes:02d}-01"
        fim_mes = (pd.Timestamp(inicio_mes) + pd.offsets.MonthEnd(1)).strftime("%Y-%m-%d")
        T2_xr = T2_xr.sel(time=slice(inicio_mes, fim_mes))

        ## PARALELIZAÇÃO
        # Criando função parcial com as variáveis
        processar_municipio_partial = partial(
                processar_municipio,
                T2_xr=T2_xr,
                outPathInter=outPathInter
            )
        
        def processar_municipio_paralelizada(mun):
            try:
                return processar_municipio_partial(mun)
            except Exception as e:
                print(f"Erro no município {mun.get('NM_MUN', 'desconhecido')}: {e}")
                return pd.DataFrame(columns=["CD_MUN", "year", "month", "day", "hour", "TEMP_C"])

        # Loop por município
        municipios = [mun for _, mun in shp_mun.iterrows()]
        with Pool(n_processadores) as p:
            final_df = pd.concat(p.map(processar_municipio_paralelizada, municipios))

        # Define o nome do arquivo CSV mensal de saída
        filename_csv = (
            f"{outPathInter}/temperatura_csv/"
            f"temperatura_cidade_{ano}_{mes:02d}.csv"
        )

        # Se o arquivo ainda não existe
        # Cria o arquivo
        final_df.to_csv(filename_csv, index=False, mode="w")

        # for _, mun in tqdm(shp_mun.iterrows(), total=len(shp_mun), desc="Processando municípios"):
            # try:
            #     processar_municipio(mun, T2_xr, outPathInter)
            # except Exception as e:
            #     print(f"Erro no município {mun.get('NM_MUN', 'desconhecido')}: {e}")
            #     continue



>>> Lendo arquivo: 1-2023


/home/artaxo/Notebooks/marcos.perrude/emiEvaporatives/scripts/preProcessor/functions/lerTemperaturaWRF.py:85: FutureWarning: updating coordinate 'time', which is a PandasMultiIndex, would leave the multi-index level coordinates ['Time', 'TSTEP'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['time', 'Time', 'TSTEP'])` to drop the coordinates' values before assigning new coordinate values.
  .assign_coords(time=time)


Erro no município Fernando de Noronha: No data found in bounds. Data variable: TEMP2


/tmp/ipykernel_1463214/2680462810.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_df = pd.concat(p.map(processar_municipio_paralelizada, municipios))



>>> Lendo arquivo: 2-2023


/home/artaxo/Notebooks/marcos.perrude/emiEvaporatives/scripts/preProcessor/functions/lerTemperaturaWRF.py:85: FutureWarning: updating coordinate 'time', which is a PandasMultiIndex, would leave the multi-index level coordinates ['Time', 'TSTEP'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['time', 'Time', 'TSTEP'])` to drop the coordinates' values before assigning new coordinate values.
  .assign_coords(time=time)


Erro no município Fernando de Noronha: No data found in bounds. Data variable: TEMP2


In [21]:
# PARA USO NO BRAVES, E PARA APROVEITAR A TEMPERATURA PRODUZIDA PARA EVAPORATIVAS POR POSTOS
# PRODUZIMOS A TEMPERATURA A PARTIR DO DADO EM GMT -3 AGREGADOS. ESSE FOI UM ERRO ACEITO E
# CONTROLADO

dataTemp = os.path.join(outPathInter, 'temperatura_csv', 'temp_media')

for i in range(1,13):
    temp = pd.read_csv("{}/temperatura_cidade_2023_{}.csv".format(f"{outPathInter}/temperatura_csv/", str(i).zfill(2)))
    medtemp = temp.groupby('CD_MUN')['TEMP_C'].mean()
    medtemp.to_csv(dataTemp + f'/2023-{i}.csv')